In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
input_path = "../../general_input/"
output_path = "../input/"
train_path = input_path + "train/"
test_path = input_path + "test/"

train_csv = train_path + "train_data_prepared.csv"
embeddings = output_path + "train_embeddings.npy"

train_split_path = output_path + "train/train_data_split.csv"
val_split_path = output_path + "val/val_data_split.csv"
train_embeddings_split = output_path + "train/train_embeddings_split.npy"
val_embeddings_split = output_path + "val/val_embeddings_split.npy"

In [3]:
df = pd.read_csv(train_csv)
embeddings = np.load(embeddings)

In [4]:
df.head()

,db,id,name,full_name,organism,taxon,gen,PE,SV,sequence,length,go_terms,aspect
0,sp,A0A023FBW4,E1142_AMBCJ,Evasin P1142,Amblyomma cajennense,34607,NaN,3,1,MTSHGAVKIAIFAVIALHSIFECLSKPQILQRTDHSTDSDWDPQMC...,97,[['GO:0019958']],['F']
1,sp,A0A023FBW7,EV546_AMBCJ,Evasin P546,Amblyomma cajennense,34607,NaN,3,1,MKVLLYIAASCLMLLALNVSAENTQQEEEDYDYGTDTCPFPVLANK...,118,[['GO:0019957']],['F']
2,sp,A0A023FDY8,EV974_AMBCJ,Evasin P974,Amblyomma cajennense,34607,NaN,1,1,MKVLLCIAASCLMLLALNVSAENTQQEEQDYDYGTDTCPFPVLANK...,118,[['GO:0019957']],['F']
3,sp,A0A023FF81,E1126_AMBCJ,Evasin P1126,Amblyomma cajennense,34607,NaN,3,1,MTSHSAVRIAIFAVIALHSIFECLSKPQILQRTDKSTDSEWDPQTC...,90,[['GO:0019958']],['F']
4,sp,A0A023FFB5,EV983_AMBCJ,Evasin P983,Amblyomma cajennense,34607,NaN,3,2,MKASFCVIASCLVVFALKGTAEDTGTEDDFDYGNTGCPFPVLGNYK...,119,[['GO:0019957']],['F']


In [5]:
# We need to use the taxon to group. Doing this, we ensure that all proteins from a specific organism are either in the train or in the validation set.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(df, groups=df['taxon']))

In [6]:
df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)

In [7]:
X_train = embeddings[train_idx]
X_val = embeddings[val_idx]

In [8]:
y_train_raw = df_train['go_terms']
y_val_raw = df_val['go_terms']

In [9]:
print(f"--- Split Summary ---")
print(f"Total proteins: {len(df)}")
print(f"Train: {len(df_train)} proteins ({len(df_train)/len(df):.1%})")
print(f"Validation: {len(df_val)} proteins ({len(df_val)/len(df):.1%})")
print(f"Organisms in Train: {df_train['taxon'].nunique()}")
print(f"Organisms in Val: {df_val['taxon'].nunique()}")

--- Split Summary ---
Total proteins: 82404
Train: 58361 proteins (70.8%)
Validation: 24043 proteins (29.2%)
Organisms in Train: 1104
Organisms in Val: 277


In [10]:
common_organisms = set(df_train['taxon']) & set(df_val['taxon'])
if len(common_organisms) == 0:
    print("OK: we have no organisms shared between train and validation sets.")
else:
    print(f"Warning: there are {len(common_organisms)} organisms shared between train and validation sets.")

OK: we have no organisms shared between train and validation sets.


In [11]:
# Save the splirts
df_train.to_csv(train_split_path, index=False)
df_val.to_csv(val_split_path, index=False)
np.save(train_embeddings_split, X_train)
np.save(val_embeddings_split, X_val)